In [1]:
%load_ext autoreload
%autoreload 2

In [14]:
import h5py, os, tqdm, glob, scipy
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.49'
import numpy as np
import matplotlib.pyplot as plt
from functools import partial

import jax
import jax.numpy as jnp
import jax_cosmo as jc

import optax
from optax.losses import huber_loss
import flax
from flax import nnx
import orbax.checkpoint as ocp
import jraph

import diffrax
from diffrax import diffeqsolve, ODETerm, LeapfrogMidpoint, PIDController, SaveAt, ConstantStepSize

import jaxpm
from jaxpm.painting import cic_paint, cic_read, compensate_cic
from jaxpm.kernels import fftk, gradient_kernel, invlaplace_kernel, longrange_kernel, invnabla_kernel
from jaxpm.utils import power_spectrum, cross_correlation_coefficients
from jaxpm.nn import MLP, CNN, HybridNet, AttentionGNN
from jaxpm import camels, plotting, hpm, nn, graph, data, diagnostics

# print(jax.devices("gpu"))
print(jax.default_backend())

gpu


# configuration

In [3]:
parts_per_dim = 64
mesh_per_dim = parts_per_dim
mesh_shape = [mesh_per_dim] * 3
box_size = [float(mesh_per_dim)] * 3

i_snapshots = np.arange(-4, 0, dtype=int)

# CAMELS

In [4]:
train_dict = camels.load_CV_snapshots(
    "CV_0",
    mesh_per_dim,
    parts_per_dim,
    i_snapshots=i_snapshots,
    return_hydro=True,
)

cosmo = train_dict["cosmo"]
scales = train_dict["scales"]

dm_poss = train_dict["dm_poss"]
dm_vels = train_dict["dm_vels"]

gas_poss = train_dict["gas_poss"]
gas_vels = train_dict["gas_vels"]

Found matching catalogs
Using snapshots ['/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/SIMBA/CV/CV_0/snapshot_084.hdf5', '/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/SIMBA/CV/CV_0/snapshot_086.hdf5', '/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/SIMBA/CV/CV_0/snapshot_088.hdf5', '/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/SIMBA/CV/CV_0/snapshot_090.hdf5']
Selecting 262144 dark matter (deterministic)
Selecting 262144 gas particles (random)


finding unique gas particle indices:  25%|██▌       | 1/4 [00:02<00:08,  2.68s/it]

Found 12711 duplicate gas particle IDs
Found 13335 duplicate gas particle IDs


finding unique gas particle indices:  50%|█████     | 2/4 [00:08<00:08,  4.39s/it]

Found 14000 duplicate gas particle IDs


...

Found 14787 duplicate gas particle IDs


finding unique gas particle indices: 100%|██████████| 4/4 [00:19<00:00,  4.89s/it]


There are 15915469 (94.86%) gas particles that exist in all snapshots


loading snapshots: 100%|██████████| 4/4 [00:53<00:00, 13.32s/it]

Could not stack h_poss
Could not stack h_masss
Could not stack h_lens
Could not stack h_ids


In [5]:
vcic_paint = jax.vmap(cic_paint, in_axes=(None,0,None))
vcic_read = jax.vmap(cic_read, in_axes=(0,0))

In [6]:
# @nnx.jit(static_argnames=("loss_fn"))
def train_step(model, optimizer, loss_fn):    
    loss, grads = nnx.value_and_grad(loss_fn)(model)
    optimizer.update(grads)

    squared_sum = jax.tree_util.tree_reduce(
            lambda x, y: x + jnp.sum(y**2),
            grads,
            0.0
        )
    grad_norm = jnp.sqrt(squared_sum)
    return loss, grad_norm

In [7]:
# @nnx.jit(static_argnames=("loss_fn", "training"))
# def train_step(model, optimizer, loss_fn, training=True):
#     def loss_wrapper(model):
#         return loss_fn(model, training=training)
    
#     loss, grads = nnx.value_and_grad(loss_wrapper)(model)
#     optimizer.update(grads)

#     squared_sum = jax.tree_util.tree_reduce(
#             lambda x, y: x + jnp.sum(y**2),
#             grads,
#             0.0
#         )
#     grad_norm = jnp.sqrt(squared_sum)
#     return loss, grad_norm

In [8]:
def solve_ode_diffrax(model, architecture, training=False):   
    ode = ODETerm(
        hpm.get_hpm_network_ode_fn(
            mesh_per_dim, 
            cosmo, 
            gravity_model=None, 
            pressure_model=model, 
            gas_architecture=architecture, 
            training=training,
        )
    )

    res = diffeqsolve(
            terms=ode,
            solver=LeapfrogMidpoint(),
            t0=scales[0],
            t1=scales[-1],
            dt0=0.04,
            y0=(dm_poss[0], dm_vels[0], gas_poss[0], gas_vels[0]),
            saveat=SaveAt(ts=scales),
            max_steps=100,
            stepsize_controller=ConstantStepSize(),
        )
    res = res.ys

    return res

# architecture

### CNN

In [9]:
# model = CNN(
#     d_in=5, 
#     d_out=1,
#     d_hidden=64,
#     n_hidden=4,
#     kernel_size=(3, 3, 3),
#     strides=1,
#     rngs=nnx.Rngs(0),
#     norm_type="batch",
#     activation=jax.nn.swish,
# )

# architecture = "cnn"

In [36]:
class debugCNN(nnx.Module):
    def __init__(
        self,
        d_in: int,
        d_hidden: int,
        d_out: int,
        kernel_size: tuple = (3, 3, 3),
        rngs: nnx.Rngs = nnx.Rngs(0),
        activation=jax.nn.relu,
    ):
        self.conv_in = nnx.Conv(d_in, d_hidden, kernel_size, strides=1, padding="CIRCULAR", rngs=rngs)
        # self.norm = nnx.BatchNorm(d_hidden, rngs=rngs)
        self.norm = nnx.LayerNorm(d_hidden, rngs=rngs)
        self.conv_out = nnx.Conv(d_hidden, d_out, kernel_size, strides=1, padding="CIRCULAR", rngs=rngs)
        
    def __call__(self, x, training: bool = False):
        if training:
            print("training mode")
        else:
            print("not training mode")
            
        x = self.conv_in(x)
        # x = self.norm(x, use_running_average=not training)
        x = self.norm(x)
        x = self.conv_out(x)

        return x

In [44]:
class debugCNN(nnx.Module):
    def __init__(
        self,
        d_in: int,
        d_hidden: int,
        d_out: int,
        kernel_size: tuple = (3, 3, 3),
        rngs: nnx.Rngs = nnx.Rngs(0),
        activation=jax.nn.relu,
    ):
        self.conv_in = nnx.Conv(d_in, d_hidden, kernel_size, strides=1, padding="CIRCULAR", rngs=rngs)
        self.norm = nnx.LayerNorm(d_hidden, rngs=rngs, feature_axes=-1)
        self.conv_out = nnx.Conv(d_hidden, d_out, kernel_size, strides=1, padding="CIRCULAR", rngs=rngs)
        
    def __call__(self, x, training: bool = False):
        if training:
            print("training mode")
        else:
            print("not training mode")
            
        x = self.conv_in(x)
        x = self.norm(x)
        x = self.conv_out(x)

        return x

In [45]:
# import flax.linen as nn

# class FlaxLinenCNN(nn.Module):
#     d_hidden: int
#     d_out: int
#     kernel_size: tuple = (3, 3, 3)
    
#     @nn.compact
#     def __call__(self, x, training: bool = False):
#         if training:
#             print("training mode")
#         else:
#             print("not training mode")

#         x = nn.Conv(features=self.d_hidden, kernel_size=self.kernel_size, padding="CIRCULAR")(x)
#         x = nn.BatchNorm(use_running_average=not training)(x)
#         x = jax.nn.relu(x)
#         x = nn.Conv(features=self.d_out, kernel_size=self.kernel_size, padding="CIRCULAR")(x)
#         return x

In [46]:
model = debugCNN(
    d_in=5, 
    d_out=1,
    d_hidden=64,
    kernel_size=(3, 3, 3),
    rngs=nnx.Rngs(0),
    activation=jax.nn.swish,
)

architecture = "cnn"

In [47]:
# model = FlaxLinenCNN(
#     d_out=1,
#     d_hidden=64,
#     kernel_size=(3, 3, 3),
# )

# architecture = "cnn"

In [48]:
res = solve_ode_diffrax(model, architecture, training=True)

dark matter and gas
Using CNN forces
Using learned pressure force
training mode
No latent variable
dark matter and gas
Using CNN forces
Using learned pressure force
training mode
No latent variable
dark matter and gas
Using CNN forces
Using learned pressure force
training mode
No latent variable


In [30]:
temp_input = np.random.random((1, 64, 64, 64, 5))
temp_output = model(temp_input, training=True)

CallCompactUnboundModuleError: Can't call compact methods on unbound modules (https://flax.readthedocs.io/en/latest/api_reference/flax.errors.html#flax.errors.CallCompactUnboundModuleError)

In [ ]:
nnx.jit()
def temp_loss_fn(model):
    temp_output = model(temp_input, training=True)
    
    return jnp.mean(temp_output**2)
    
loss, grads = nnx.value_and_grad(temp_loss_fn)(model)